# DevTutor Bot: LoRA fine-tuning

Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with LoRA so it answers like DevTutor Bot: in Spanish, concise, only from the retrieved Angular/TypeScript docs, and saying so when the docs don't cover a question.

The fine-tuning teaches **style and behavior**; the facts still come from RAG at answer time. Each training example is exactly the prompt the API sends (docs context + question) followed by the answer we want, written by a larger teacher model (`scripts/generate_dataset.py`).

Steps (same as the course practice):
1. Load the dataset
2. Load the base model and test it (before)
3. Configure LoRA
4. Train with `SFTTrainer`
5. Test again (after) and compare
6. Save the adapter and use it in the API

**Runtime:** Runtime > Change runtime type > **T4 GPU**. Training takes about 10–15 minutes.

In [ ]:
!pip install -q -U transformers peft trl accelerate datasets

## 1. Load the dataset

Downloads `data/devtutor_sft.jsonl` from the GitHub repo. If the repo isn't public or the file isn't pushed yet, upload it with the file picker instead.

In [ ]:
import os
from datasets import load_dataset

DATASET_URL = "https://raw.githubusercontent.com/mcastig/api-dev-tutor-bot/main/data/devtutor_sft.jsonl"
DATASET_FILE = "devtutor_sft.jsonl"

if not os.path.exists(DATASET_FILE):
    !wget -q -O {DATASET_FILE} {DATASET_URL} || rm -f {DATASET_FILE}
if not os.path.exists(DATASET_FILE):
    from google.colab import files
    print("Download failed; upload data/devtutor_sft.jsonl from the repo:")
    files.upload()

dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

# Hold out 10% to compare the model before and after training.
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"].select_columns(["prompt", "completion"])
test_dataset = split["test"]
print(f"Train: {len(train_dataset)}  Test: {len(test_dataset)}")
print(dataset[0]["prompt"][0]["content"])
print("---")
print(dataset[0]["completion"][0]["content"])

## 2. Load the base model and test it (before)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(42)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# T4 GPUs don't support bfloat16; newer GPUs (A100, L4) do.
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=dtype, device_map="auto")
print("Base model loaded:", BASE_MODEL)

In [ ]:
def generate_answer(model_to_use, prompt, max_new_tokens=300):
    """Same generation as the API's local mode (greedy decoding)."""
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model_to_use.device)
    output = model_to_use.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# A few held-out examples, including an off-topic one if the split has it.
examples = [test_dataset[i] for i in range(3)]
off_topic = [e for e in test_dataset if e["tipo"] == "fuera_de_tema"]
if off_topic:
    examples.append(off_topic[0])

answers_before = []
for example in examples:
    answer = generate_answer(model, example["prompt"][0]["content"])
    answers_before.append(answer)
    print("Question:", example["pregunta"])
    print("Base model:", answer, "\n")

## 3. Configure LoRA

LoRA trains small low-rank matrices next to the model's weights instead of the weights themselves, so only ~1% of the parameters change.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # All attention and MLP projections: more capacity than q/v only, still small.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

## 4. Train with LoRA

The dataset is in "prompt-completion" format, so `SFTTrainer` only computes the loss on the answers: the model learns to *answer*, not to repeat the docs.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir="/content/results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,  # float = ratio of total steps
    logging_steps=5,
    max_length=1024,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none",
)

trainer = SFTTrainer(
    model=lora_model,
    train_dataset=train_dataset,
    args=training_config,
    processing_class=tokenizer,
)

training_result = trainer.train()
losses = [log["loss"] for log in trainer.state.log_history if "loss" in log]
print(f"Loss at the start: {losses[0]:.2f}")
print(f"Loss at the end: {losses[-1]:.2f}")
print(f"Reduction: {(1 - losses[-1] / losses[0]) * 100:.0f}%")

## 5. Test again (after) and compare

In [ ]:
lora_model.eval()
for example, before in zip(examples, answers_before):
    print("Question:", example["pregunta"])
    print("Before:", before)
    print("After:", generate_answer(lora_model, example["prompt"][0]["content"]))
    print("Teacher (target):", example["completion"][0]["content"], "\n")

## 6. Save the adapter and use it in the API

The adapter is only a few tens of MB; the API downloads the base model itself. `training_info.json` records how it was trained, and the API's `/api/info` endpoint shows it.

1. Download `devtutor-lora.zip` (next cell) and unzip it into the repo as `models/devtutor-lora/` (it's git-ignored).
2. Start the API in local mode:

```bash
uv sync --group finetuned
USE_FINETUNED=1 uv run fastapi dev
```

In [ ]:
import json, peft, transformers, trl

ADAPTER_DIR = "/content/devtutor-lora"
lora_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Training details, shown by the API's /api/info endpoint.
trainable, total = lora_model.get_nb_trainable_parameters()
training_info = {
    "dataset": {
        "file": "data/devtutor_sft.jsonl",
        "train_examples": len(train_dataset),
        "test_examples": len(test_dataset),
        "off_topic_examples": sum(1 for e in dataset if e["tipo"] == "fuera_de_tema"),
        "format": "prompt-completion (loss on the answer only)",
    },
    "hyperparameters": {
        "epochs": training_config.num_train_epochs,
        "batch_size": training_config.per_device_train_batch_size,
        "gradient_accumulation_steps": training_config.gradient_accumulation_steps,
        "effective_batch_size": training_config.per_device_train_batch_size
        * training_config.gradient_accumulation_steps,
        "learning_rate": training_config.learning_rate,
        "lr_scheduler": str(training_config.lr_scheduler_type.value),
        "warmup_steps": training_config.warmup_steps,
        "max_length": training_config.max_length,
        "precision": "bf16" if use_bf16 else "fp16",
        "seed": 42,
    },
    "results": {
        "steps": trainer.state.global_step,
        "initial_loss": round(losses[0], 4),
        "final_loss": round(losses[-1], 4),
        "loss_reduction_percent": round((1 - losses[-1] / losses[0]) * 100, 1),
        "runtime_seconds": round(training_result.metrics["train_runtime"]),
    },
    "trainable_parameters": trainable,
    "total_parameters": total,
    "hardware": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "libraries": {
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "trl": trl.__version__,
        "torch": torch.__version__,
    },
}
with open(f"{ADAPTER_DIR}/training_info.json", "w") as f:
    json.dump(training_info, f, indent=2)
print(json.dumps(training_info, indent=2))

!cd /content && zip -qr devtutor-lora.zip devtutor-lora
from google.colab import files
files.download("/content/devtutor-lora.zip")

In [ ]:
# Optional: push the adapter to your Hugging Face account instead of
# downloading it (needs a *write* token), then set
# FINETUNED_ADAPTER=your-user/devtutor-lora in src/.env.
# from huggingface_hub import notebook_login
# notebook_login()
# lora_model.push_to_hub("your-user/devtutor-lora")
# tokenizer.push_to_hub("your-user/devtutor-lora")